# LET OP DEZE OPDRACHT IS NOOIT AFGEROND
Dit is een van de twee niet geslaagde pogingen om een neuraal netwerk te implementeren.

## Zelf een NN maken zonder library

We gaan nu zelf een Neuraal NEtwerk implementeren zonder gebruik te maken van libraries voor Neurale Netwerken. Sklearn gebruiken om de mnsit data te laden en numpy gebruiken mag uiteraard wel.

### Data laden

Laad de mnist data zoals je het eerder hebt geladen. Split dedara op in een test/training set. Bepaal zelf op hoeveel je split.

Normaliseer de data op 0-1 ipv de pixelwaardes. Dit heb je ook al eerder gedaan. 

Doe een one hot encoding op de labels van de data.

In [1]:
import numpy as np;
import random as rnd;
import math;
from tensorflow.keras.datasets import mnist;

In [2]:
def OneHotEncodeDigit(label: int):
    """
    One hot encode a single one digit label
    """
    arr = np.zeros((10))
    arr[label] = 1
    return arr

def OneHotEncode(arr):
    """
    One hot encode an array of labels, like the train_images
    """
    return np.array([OneHotEncodeDigit(item) for item in arr])

def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum

    return normalised.astype(np.half)

In [3]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = np.array([normalize_and_vectorize_image(img) for img in train_images])
train_labels_encoded = OneHotEncode(train_labels)
# train_labels = np.array([np.array([int(n % 2 == 0)]) for n in train_labels])
test_images = np.array([normalize_and_vectorize_image(img) for img in test_images])
test_labels_encoded = OneHotEncode(test_labels)
# test_labels = np.array([np.array([int(n % 2 == 0)]) for n in test_labels])


In [ ]:
print(test_labels)

### Activatie functies

Maak de volgende activatiefuncties: 

    def relu(waardes)

en 

    def softmax(waardes) 

Als het goed is kun je ze uit de vorige opdracht gebruiken. 

In [ ]:
def relu(values: list):
    return np.array([max(0, value) for value in values])

def relu_derivative(x):
    return np.array([1 if y > 0 else 0 for y in x])
    #return 1 if x > 0 else 0


### Cross entropy loss

Maak een functie

    def cross_entropy(y_true, y_predicted)

functie om de grootte van loss te kunnen berekenen. We gaan de waarde van de loss gebruiken als validatie output. Het wordt dus aan het einde geprint.

In [ ]:
def cross_entropy(true, predicted):
    # N = predicted.shape[0]
    # loss = -1 / N * np.sum(true * np.log(predicted) + (1 - true) * np.log(1 - predicted))
    # return loss
    return -np.sum(true * np.log(predicted))

### Initialiseer je NN 

Maak de volgende variabelen:

- Kies hoeveel input nodes je hebt. 
- En hoeveel hidden layer nodes? 
- En hoeveel output nodes? Leg uit waarom je dat hebt gekozen.


Maak voor nu 1 input layer en 1 hidden layer. JE mag dit later uitbreiden

Om de initiele gewichten een random waarde te geven kun je np.random.randn gebruiken. Deze geeft een normale verdeling met waarden die ongeveer liggen tussen -3 en +3.

Om grote getallen te voorkomen kun je deze weer kleiner maken door bijv *0.01 te doen, dan wordt de standaardafwijking 0.01

In [ ]:
W1 = np.random.randn(784, 16) * 0.1
b1 = np.random.randn(16) * 0.1

W2 = np.random.randn(16, 10) * 0.1
b2 = np.random.randn(10) * 0.1

### Forward pass
Maak een forward propagation functie. We returnen ook tussenwaarden (cache) zodat we die voor backpropagation runnen gebruiken.

#### Stap 1

Maak functie def forward()
Als parameters hgebruik de volgende:
- inputdata X
- gewichtsmatrix W1 van de hidden layer nodes
- biasvector b1 van de hidden layer nodes
- gewichtsmatrix W2 van de output layer nodes
- biasvector b2 van de output layer nodes

Bereken eerst de invoer van de hidden layer. Vermenigvuldig de input met de gewichten en tel de bias erbij op.


Hint:

denk aan matrixvermenigvuldiging
de vorm moet kloppen: (samples, features) @ (features, hidden)

#### Stap 2

De uitkomst van stap 1 is de ruwe activatie van de hidden layer. Pas nu de door jou geiplementeerde ReLU-functie hierop toe.

#### Stap 3

Gebruik de output van de hidden layer als input voor de outputlaag. Vermenigvuldig met W2 en tel de bias b2 er bij op. Eigenlijk bijna hetzelfde als wat je in stap 1 hebt gedaan maar met andere dimensies.

#### Stap 4

De output van stap 3 zijn “scores” (logits), nog geen kansen. Zet deze scores om naar kansen met de softmax-functie die je eerder hebt gemaakt.

#### Stap 5


Voor de backpropagation stap heb je straks de tussenwaarden nodig.
Sla alle relevante variabelen op in één object (bijv. tuple)

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

#### Stap 6

return nu (bijvoorbeeld in een tuple vorm) de voorspelling en de object uit Stap 5

In [ ]:
def forward(X_train, W1, b1, W2, b2):
    result = X_train @ W1
    result += b1
    result = relu(result)

    result = result @ W2
    result += b2
    result = softmax(result)
    print(result)

In [9]:
forward(train_images[0], W1, b1, W2, b2)

NameError: name 'softmax' is not defined

### Backward pass

We gaan enkele functies voorbereiden die we in de les hebben gehad om de backpropagation stap uit te voeren.

We beginnen met berekening van de gradient van loss.

Tip: Je kunt de formule uit de slides van deze week gebruiken. 

Maak nu een methode 
        
        def compute_output_gradient(final_prediction, correct_answers)
        
die de output (laag) gradient teruggeeft. Je kunt de formule hierboven gebruiken.


In [ ]:
def compute_output_gradient(final_prediction, correct_anwsers):
    return final_prediction - correct_anwsers

Maak nu methode 


    def compute_output_gradients(hidden_output, output_gradient)

die de gradienten van de output‑gewichten teruggeeft. Voeg aan de output van je methode ook de gradient van bias toe (als tuple bijv.). 

Tip: Je kunt de formule uit de slides van deze week gebruiken.


In [ ]:
def compute_output_gradients(hidden_output, output_gradient):
    return hidden_output @ output_gradient

def compute_bias_gradient(output_gradient):
    return output_gradient

Maak de methode 

    def compute_hidden_gradient(output_gradient, hidden_to_output_weights)
    
Deze zegt hoe sterk elke hidden neuron bijdraagt aan de fout.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [ ]:
def compute_hidden_gradient(output_gradient: np.ndarray, hidden_to_output_weights):
    return output_gradient.sum(axis=0) @ hidden_to_output_weights

Schrijf een python functie die afgeleide RELU berekent voor een getal:

    def relu_derivative(x)

Pas het aan zodat het voor een lijst van getallen werkt (eg met numpy)


Maak nu methode 

    def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)

De functie moet de gradienten van de hidden-laag gewichten en bias teruggeven. Je hebt de afgeleide van de RELU nodig uit de vorige opgave, want alleen die gewichten doen mee.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [ ]:
def compute_hidden_gradients(hidden_gradient, hidden_raw_gradients, input_data):
    return relu_derivative(input_data @ hidden_gradient) @ hidden_gradient

Maak nu de backward propagation nmethode die de bovenstaatde methodes zal gbruiken.

    def backward(y_true, cache, W2):

De parameter cache zal uit de forward pass komen en als het goed is de volgende waardes bevatten:

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

Die kun je dus daaruit unpacken.

Doe nu de volgende stappen achter elkaar:

- Bereken de output gradient (gebruik je eerder gemaakte methode)
- Bereken de output gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en output gradient)

- Bereken de hidden gradient (gebruik je eerder gemaakte methode)
- Bereken de hidden gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en hidden gradient)

return de output gradients en de hidden gradients bijvoorbeeld in de vorm van een tuple:

    return dW1, db1, dW2, db2

In [ ]:
# def backward(y_true, cache, W2):


### De training loop. 

JE mag het volgende stukje code gebruiken om je netwerk te trainen. MAar je mag uiteraard ook zelf een andere opzet maken.


One hot encoding toepassen (op de labels)

### Experimenteer

Werkt het? Mooi, dan kun je nu zelf experimenteren met verschillende setup parameters en kijken wat het beste werkt.

In [26]:
class NeuralNet():
    '''
    A two layer neural network
    '''

    def __init__(self, layers=[784,128,10], learning_rate=0.001, iterations=100, batch_size=32):
        self.params = {}
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.loss = []
        self.sample_size = None
        self.layers = layers
        self.X = None
        self.y = None
        self.batch_size = batch_size

    def init_weights(self):
        '''
        Initialize the weights from a random normal distribution
        '''
        np.random.seed(1) # Seed the random number generator
        self.params["W1"] = np.random.randn(self.layers[0], self.layers[1]) * 0.01
        self.params['b1']  =np.random.randn(self.layers[1],) * 0.01
        self.params['W2'] = np.random.randn(self.layers[1],self.layers[2]) * 0.01
        self.params['b2'] = np.random.randn(self.layers[2],) * 0.01

    def relu(self,Z):
        '''
        The ReLu activation function is to performs a threshold
        operation to each input element where values less
        than zero are set to zero.
        '''
        return np.maximum(0,Z)

    def dRelu(self, x):
        x[x<=0] = 0
        x[x>0] = 1
        return x

    def eta(self, x):
      ETA = 0.0000000001
      return np.maximum(x, ETA)


    def sigmoid(self,Z):
        '''
        The sigmoid function takes in real numbers in any range and
        squashes it to a real-valued output between 0 and 1.
        '''
        return 1/(1+np.exp(-Z))

    def softmax(self, x: np.ndarray):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()


    def entropy_loss(self,y, yhat):
        nsample = len(y)
        yhat_inv = 1.0 - yhat
        y_inv = 1.0 - y
        yhat = self.eta(yhat) ## clips value to avoid NaNs in log
        yhat_inv = self.eta(yhat_inv)
        loss = -1/nsample * (np.sum(np.multiply(np.log(yhat), y) + np.multiply((y_inv), np.log(yhat_inv))))
        return loss

    def forward_propagation(self):
        '''
        Performs the forward propagation
        '''

        Z1 = self.X.dot(self.params['W1']) + self.params['b1']
        A1 = self.relu(Z1)
        Z2 = A1.dot(self.params['W2']) + self.params['b2']
        yhat = self.sigmoid(Z2)
        loss = self.entropy_loss(self.y,yhat)

        # save calculated parameters
        self.params['Z1'] = Z1
        self.params['Z2'] = Z2
        self.params['A1'] = A1

        return yhat,loss

    def back_propagation(self,yhat):
        '''
        Computes the derivatives and update weights and bias according.
        '''
        y_inv = 1 - self.y
        yhat_inv = 1 - yhat

        dl_wrt_yhat = np.divide(y_inv, self.eta(yhat_inv)) - np.divide(self.y, self.eta(yhat))
        dl_wrt_sig = yhat * (yhat_inv)
        dl_wrt_z2 = dl_wrt_yhat * dl_wrt_sig

        dl_wrt_A1 = dl_wrt_z2.dot(self.params['W2'].T)
        dl_wrt_w2 = self.params['A1'].T.dot(dl_wrt_z2)
        dl_wrt_b2 = np.sum(dl_wrt_z2, axis=0, keepdims=True)

        dl_wrt_z1 = dl_wrt_A1 * self.dRelu(self.params['Z1'])
        dl_wrt_w1 = self.X.T.dot(dl_wrt_z1)
        dl_wrt_b1 = np.sum(dl_wrt_z1, axis=0, keepdims=True)

        #update the weights and bias
        self.params['W1'] = self.params['W1'] - self.learning_rate * dl_wrt_w1
        self.params['W2'] = self.params['W2'] - self.learning_rate * dl_wrt_w2
        self.params['b1'] = self.params['b1'] - self.learning_rate * dl_wrt_b1
        self.params['b2'] = self.params['b2'] - self.learning_rate * dl_wrt_b2

    def fit(self, X, y, verbose=0):
        '''
        Trains the neural network using the specified data and labels
        '''
        self.init_weights() #initialize weights and bias

        x_len = X.shape[0] - self.batch_size

        for i in range(self.iterations):
            for batch_nr in range(x_len // self.batch_size):
                batch_start = batch_nr * self.batch_size
                batch_end = (batch_nr + 1) * self.batch_size

                self.X = X[batch_start:batch_end]
                self.y = y[batch_start:batch_end]

                yhat, loss = self.forward_propagation()

                self.back_propagation(yhat)
                self.loss.append(loss)
            if verbose == 1:
                print(f"Epoch {i} finished, Loss: {self.loss[-1]}")


    def predict(self, X):
        '''
        Predicts on a test data
        '''
        Z1 = X.dot(self.params['W1']) + self.params['b1']
        A1 = self.relu(Z1)
        Z2 = A1.dot(self.params['W2']) + self.params['b2']
        pred = self.sigmoid(Z2)
        return np.argmax(pred)
        # return np.round(pred)

    def acc(self, y, yhat):
        '''
        Calculates the accutacy between the predicted valuea and the truth labels
        '''
        acc = int(sum(y == yhat) / len(y) * 100)
        return acc


    def plot_loss(self):
        '''
        Plots the loss curve
        '''
        plt.plot(self.loss)
        plt.xlabel("Iteration")
        plt.ylabel("logloss")
        plt.title("Loss curve for training")
        plt.show()

In [27]:
nn = NeuralNet(iterations=10, learning_rate=0.001)

nn.fit(train_images, train_labels_encoded, verbose=1)

correct = 0
fails = dict()
for i in range(10):
    fails[i] = 0
for i, _ in enumerate(test_images):
    predict = nn.predict(test_images[i])
    print(predict, "was", test_labels[i])
    if np.array_equal(predict, test_labels[i]):
        correct += 1
    else :
        fails[predict] += 1

print("Correct count:", correct)
print(fails)

Epoch 0 finished, Loss: 0.47338557237267315
Epoch 1 finished, Loss: 0.31845988044751866
Epoch 2 finished, Loss: 0.2393428205740528
Epoch 3 finished, Loss: 0.19525663666827187


KeyboardInterrupt: 